# Antigen vs antigen scatters

Replicate-averaged CH505TF affinity against SF162 affinity for every measured genotype of one antibody, colored by which of a chosen set of sites it carries. Genotypes at the assay floor are moved into a jittered separated band below the threshold. The germline and fully mature genotypes are marked, hollow when there was no assay measurement and model inferred values is provided as replacement.

Any antibody and any combination of its ten sites can be passed; the batch cell at the end constructs the four figures that appear in the paper.

## Imports

In [ ]:
import os, glob, itertools, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

plt.rcParams.update({"font.family":"DejaVu Sans","font.size":9,"axes.linewidth":0.9,
    "mathtext.default":"regular","xtick.major.size":3.5,"ytick.major.size":3.5,
    "xtick.major.width":0.9,"ytick.major.width":0.9,
    "svg.fonttype":"path", "pdf.fonttype":42})

## Configuration

In [ ]:
antibody     = "VRC01"
select_sites = ["N52K", "S74Y"]

data_dir = "../data"
fig_dir  = "../figures"
os.makedirs(fig_dir, exist_ok=True)
png_dpi  = 300

name_to_stem = {"1NC9":"1NC9", "CH235.12":"CH235_12", "N49P7":"N49P7",
                "NIH45-46":"NIH4546", "VRC01":"VRC01", "VRC07":"VRC07",
                "VRC13":"VRC13", "VRC16":"VRC16", "VRC-PG20":"VRCPG20"}

sites     = [f"site_{i}" for i in range(10)]
antigens  = ["CH505TF", "SF162"]      # x antigen second, y antigen first
floor_kd  = 6.0                        # assay detection floor
hi_step   = 0.5                        # round the top of the range up to this step
censored_frac = 1/6                    # share of the plotted range sitting below the floor
jitter_frac   = 0.14                   # censored jitter, as a share of the censored band

# eight background colors, indexed by subset in size then panel order
bg_hex8 = ["#6b5894","#002792","#8eb4fb","#386a4f","#cfb454","#b96040","#d48da6","#741345"]
# two-site figures use four of them, so the light blue to maroon ramp still reads
bg_hex4 = [bg_hex8[i] for i in (2, 3, 4, 7)]

point_size    = 16     # scatter points
germline_size = 200    # filled when measured, white filled when inferred
mature_size   = 430

lab_meas = r"Measured Binding Affinity ($-\log_{10}K_D$)"
lab_floor = "\u2264" + f"{floor_kd:.1f}"

# reference genotypes and the marker each one gets
ref_points = [("G"*10, "o", "germline"), ("M"*10, "*", "mature")]

## Data handling

In [ ]:
# Display name to file stem, erroring with the known names when one is wrong
def stem_of(ab):
    if ab in name_to_stem: return name_to_stem[ab]
    raise KeyError(f"unknown antibody {ab!r}; known: {sorted(name_to_stem)}")

# Kabat label to site index, in site order
def site_labels(ab):
    m = pd.read_csv(f"{data_dir}/{stem_of(ab)}_meta.csv")
    m = m.assign(idx=m["site"].str.replace("site_","",regex=False).astype(int)).sort_values("idx")
    return {f"{r.germline_aa}{r.kabat_position}{r.mature_aa}": r.idx for r in m.itertuples()}

# Site indices for the requested labels, erroring with the real options when one is wrong
def resolve_sites(ab, wanted):
    n2s = site_labels(ab)
    bad = [s for s in wanted if s not in n2s]
    if bad:
        raise KeyError(f"{ab}: no site {bad}; available are {list(n2s)}")
    return [n2s[s] for s in wanted]

# Inferred landscape table for one antibody and antigen
def df_all(ab, ag):
    return pd.read_csv(f"{data_dir}/{stem_of(ab)}_{ag}_BEST_df_all.csv").drop_duplicates("genotype_tokens")

# One row per variant measured against both antigens, replicate-averaged, plus its token
def variant_xy(ab):
    k = pd.read_csv(f"{data_dir}/{stem_of(ab)}_kds.csv")
    k["tok"] = k[sites].agg("".join, axis=1)
    def avg(ag):
        d = k[k.antigen == ag]
        return d.groupby("id", as_index=False).agg(v=("nlog10_Kd","mean"), tok=("tok","first"))
    y = avg(antigens[0]).rename(columns={"v":"y"})
    x = avg(antigens[1]).rename(columns={"v":"x"})
    return y.merge(x[["id","x"]], on="id")

# Every subset of the selected sites, empty first then by size
def subsets(labels):
    return [tuple(c) for r in range(len(labels)+1) for c in itertools.combinations(labels, r)]

# Colors for n subsets: the four-color ramp at four, the full eight at eight, else cycle
def palette_for(n):
    if n == len(bg_hex4): return list(bg_hex4)
    if n == len(bg_hex8): return list(bg_hex8)
    return [bg_hex8[i % len(bg_hex8)] for i in range(n)]

## Drawing
Reference genotypes are filled when measured and white filled when the value comes from the Tobit fit.

In [ ]:
# Top rounded up to hi_step, bottom leaving room for the censored band
def axis_range(m):
    hi = float(np.ceil(max(m["x"].max(), m["y"].max()) / hi_step) * hi_step)
    lo = floor_kd - (hi - floor_kd) * censored_frac
    return lo, hi

# Move censored values into a gap-separated band below the floor
def cluster_floor(v, lo, rng):
    v = np.asarray(v, float).copy(); f = np.isclose(v, floor_kd)
    band = floor_kd - lo
    v[f] = (lo + floor_kd)/2 + rng.uniform(-jitter_frac, jitter_frac, f.sum()) * band
    return v

# Curly brace just outside an axis, data coords along it and axes coords across it
def brace(ax, span, which, depth=0.030, label=None, at=-0.014, pad=0.012):
    lo, hi = span
    n = 101; half = n // 2 + 1
    t = np.linspace(lo, hi, n); th = t[:half]
    beta = 44.0 / max(hi - lo, 1e-9)
    b = 1/(1+np.exp(-beta*(th-th[0]))) + 1/(1+np.exp(-beta*(th-th[-1])))
    b = np.concatenate((b, b[-2::-1])); b = b - b.min(); b = b / max(b.max(), 1e-9)
    off = at - depth*b
    tr = ax.get_xaxis_transform() if which == "x" else ax.get_yaxis_transform()
    xs, ys = (t, off) if which == "x" else (off, t)
    ax.plot(xs, ys, transform=tr, color="black", lw=1.1, clip_on=False, zorder=6,
            solid_capstyle="round")
    if label:
        if which == "x":
            ax.text((lo+hi)/2, at-depth-pad, label, transform=tr, ha="center", va="top",
                    fontsize=10, clip_on=False)
        else:
            ax.text(at-depth-pad, (lo+hi)/2, label, transform=tr, ha="right", va="center",
                    fontsize=10, clip_on=False)

# (Token, marker, measured, label) per reference genotype
def reference_state(ab, m):
    out = []
    for tok, mark, name in ref_points:
        measured = bool(len(m[m["tok"] == tok]))
        out.append((tok, mark, measured, name if measured else f"{name} (inferred)"))
    return out

# (x, y) predicted affinity for a genotype with no paired measurement
def inferred_xy(ab, tok):
    out = []
    for ag in (antigens[1], antigens[0]):
        d = df_all(ab, ag)
        out.append(float(d.loc[d["genotype_tokens"].astype(str) == tok, "predicted_kd_full"].iloc[0]))
    return out

# One scatter, points colored by which of the selected sites are mature
def draw_scatter(ax, ab, sel, m, lo, hi):
    rng = np.random.default_rng(0)
    subs = subsets(sel); cols = palette_for(len(subs))
    ax.plot([lo,hi],[lo,hi], ls="--", color="#B0B0B0", lw=1.0, zorder=0)
    ax.axhline(floor_kd, ls=":", color="#d1a0a0", lw=1.0, zorder=0)
    ax.axvline(floor_kd, ls=":", color="#d1a0a0", lw=1.0, zorder=0)
    for i, sub in enumerate(subs):
        d = m[m["grp"] == i]
        if not len(d): continue
        ax.scatter(cluster_floor(d["x"], lo, rng), cluster_floor(d["y"], lo, rng),
                   s=point_size, color=cols[i], alpha=.85, edgecolors="none", zorder=2)
    for tok, mark, measured, _ in reference_state(ab, m):
        size = germline_size if mark == "o" else mature_size
        d = m[m["tok"] == tok]
        if measured:
            ax.scatter(cluster_floor(d["x"], lo, rng), cluster_floor(d["y"], lo, rng),
                       s=size, marker=mark, c="black", zorder=5)
        else:
            x, y = inferred_xy(ab, tok)
            ax.scatter(cluster_floor([x], lo, rng), cluster_floor([y], lo, rng), s=size,
                       marker=mark, facecolors="white", edgecolors="black", linewidths=1.8, zorder=5)
    ticks = np.arange(np.ceil((floor_kd+0.25)/hi_step)*hi_step, hi+1e-9, hi_step)
    ax.set_xlim(lo,hi); ax.set_ylim(lo,hi); ax.set_aspect("equal")
    ax.set_xticks(ticks); ax.set_xticklabels([f"{t:.1f}" for t in ticks], fontsize=10)
    ax.set_yticks(ticks); ax.set_yticklabels([f"{t:.1f}" for t in ticks], fontsize=10)
    band = ((lo+floor_kd)/2 - (floor_kd-lo)*0.35, (lo+floor_kd)/2 + (floor_kd-lo)*0.35)
    brace(ax, band, "x", label=lab_floor)
    brace(ax, band, "y", label=lab_floor)
    ax.set_xlabel(f"{antigens[1]}\n"+lab_meas, fontsize=11, labelpad=18)
    ax.set_ylabel(f"{antigens[0]}\n"+lab_meas, fontsize=11)

# Marker note, color strip, genotype dot grid and n row
def draw_key(ax, ab, sel, m, strip_frac=0.72):
    subs = subsets(sel); cols = palette_for(len(subs))
    ncol = len(subs); nrow = len(sel)
    x0 = (1 - strip_frac) / 2; w = strip_frac / ncol
    refs = reference_state(ab, m)
    spots = [0.26, 0.58] if len(refs) == 2 else np.linspace(0.16, 0.78, len(refs))
    for (tok, mark, measured, label), sx in zip(refs, spots):
        # stars sit wider than circles at the same area, so they need a bigger text gap
        size, gap = (germline_size*0.55, 0.030) if mark == "o" else (mature_size*0.55, 0.048)
        fill = dict(color="black") if measured else dict(facecolors="white", edgecolors="black", linewidths=1.6)
        ax.scatter([sx],[nrow+1.5], s=size, marker=mark, clip_on=False, zorder=4, **fill)
        ax.text(sx+gap, nrow+1.5, label, ha="left", va="center", fontsize=11)
    for i, c in enumerate(cols):
        ax.add_patch(plt.Rectangle((x0+i*w, nrow+0.30), w, 0.42, facecolor=c, edgecolor="none", zorder=3))
    for r, name in enumerate(sel):
        yy = nrow - 1 - r
        ax.text(x0-0.035, yy, name, ha="right", va="center", fontsize=11)
        for i, sub in enumerate(subs):
            ax.scatter(x0+(i+0.5)*w, yy, s=42, color="#222222" if name in sub else "#e6e6e6", zorder=3)
    ax.text(x0-0.035, -1.0, "n=", ha="right", va="center", fontsize=11)
    for i in range(ncol):
        ax.text(x0+(i+0.5)*w, -1.0, str(int((m["grp"] == i).sum())), ha="center", va="center", fontsize=11)
    ax.set_xlim(0,1); ax.set_ylim(-1.7, nrow+2.2); ax.axis("off")

# Antigen vs antigen scatter for one antibody, colored by the selected sites
def build_scatter(ab, sel, save=True):
    idx = resolve_sites(ab, sel)
    m = variant_xy(ab)
    subs = subsets(sel)
    m["grp"] = m["tok"].apply(lambda t: subs.index(tuple(s for s, i in zip(sel, idx) if t[i] == "M")))
    lo, hi = axis_range(m)
    fig = plt.figure(figsize=(5.8, 6.6))
    gs = GridSpec(2, 1, figure=fig, height_ratios=[1, 0.40], hspace=0.34,
                  left=0.17, right=0.97, top=0.98, bottom=0.03)
    draw_scatter(fig.add_subplot(gs[0]), ab, sel, m, lo, hi)
    draw_key(fig.add_subplot(gs[1]), ab, sel, m)
    if save:
        stem = f"{fig_dir}/{stem_of(ab).lower()}_scatter_{'_'.join(sel)}"
        for ext in ("png","pdf","svg"):
            fig.savefig(f"{stem}.{ext}", bbox_inches="tight", **({"dpi": png_dpi} if ext == "png" else {}))
        assert open(stem+".svg").read().count("<text") == 0, "live text present, set svg.fonttype='path'"
        print(f"saved {stem}.png / .svg / .pdf   n={len(m)}  range {lo:.2f} to {hi:.2f}")
    return fig

## Generate

In [ ]:
build_scatter(antibody, select_sites)

## Batch
Any set of antibody and site combinations in one pass.

In [ ]:
wanted = [
    ("VRC01",    ["N52K", "S74Y"]),
    ("N49P7",    ["M34I", "S54G"]),
    ("VRC-PG20", ["I51M", "S54W"]),
    ("VRC07",    ["H35N", "N52K", "S54F"]),
]

for ab, sel in wanted:
    build_scatter(ab, sel)
    plt.close("all")

## Available site labels

In [ ]:
for ab in name_to_stem:
    print(f"{ab:10} {list(site_labels(ab))}")